<a href="https://colab.research.google.com/github/EngMohamed-op/supervised-and-unsupervised-project/blob/main/WorldCup2026Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from google.colab import files


In [2]:
uploaded = files.upload()

Saving fifa_ranking-2024-06-20.csv to fifa_ranking-2024-06-20.csv
Saving results.csv to results.csv


In [ ]:


# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
print("Loading datasets...")
df_results = pd.read_csv('results.csv')
df_fifa = pd.read_csv('fifa_ranking.csv')

# Convert dates to datetime objects
df_results['date'] = pd.to_datetime(df_results['date'])
df_fifa['rank_date'] = pd.to_datetime(df_fifa['rank_date'])

# Clean string data (remove leading/trailing whitespaces)
df_results['home_team'] = df_results['home_team'].str.strip()
df_results['away_team'] = df_results['away_team'].str.strip()
df_fifa['country_full'] = df_fifa['country_full'].str.strip()

# ==========================================
# 2. TEAM NAME MAPPING
# Ensures consistency between match results and FIFA rankings
# ==========================================
name_mapping = {
    'USA': 'United States',
    'Korea Republic': 'South Korea',
    'IR Iran': 'Iran',
    'Korea DPR': 'North Korea',
    'China PR': 'China',
    'Czechia': 'Czech Republic',
    'Türkiye': 'Turkey',
    'Ivory Coast': "Côte d'Ivoire",
    'DR Congo': 'Congo DR',
    'Cape Verde': 'Cabo Verde',
    'Kyrgyzstan': 'Kyrgyz Republic',
    'Vietnam': 'Viet Nam'
}
df_results['home_team'] = df_results['home_team'].replace(name_mapping)
df_results['away_team'] = df_results['away_team'].replace(name_mapping)

# Filter data from 2002 onwards and sort chronologically
df_results = df_results[df_results['date'] >= '2002-01-01'].sort_values('date').reset_index(drop=True)

# ==========================================
# 3. INITIAL SEEDING (Start of 2002)
# Using real FIFA points as the baseline for all teams existing in 2002
# ==========================================
print("Extracting real 2002 baseline points...")
fifa_2002 = df_fifa[df_fifa['rank_date'].dt.year == 2002].sort_values('rank_date').groupby('country_full').first()
elo_ratings = fifa_2002['total_points'].to_dict()

# ==========================================
# 4. HELPER FUNCTIONS
# ==========================================

# Function to fetch points for "New Teams" at their first historical appearance
def fetch_initial_points(team_name, debut_date):
    """Searches FIFA rankings for a team's points at the time of their debut."""
    first_record = df_fifa[(df_fifa['country_full'] == team_name) &
                           (df_fifa['rank_date'] >= debut_date)].sort_values('rank_date')
    if not first_record.empty:
        return first_record.iloc[0]['total_points']
    return 1400  # Default fallback if no FIFA record exists

# Function to determine Tournament Importance (K-Factor)
def get_k_factor(tournament):
    if 'FIFA World Cup' in tournament and 'Qualification' not in tournament: return 60
    if 'Euro' in tournament or 'Copa América' in tournament or 'Asian Cup' in tournament: return 40
    if 'Qualification' in tournament: return 30
    return 20 # Friendlies and minor tournaments

# Function for Goal Margin Factor (G-Factor)
def get_goal_margin_factor(h_s, a_s):
    diff = abs(h_s - a_s)
    if diff <= 1: return 1
    if diff == 2: return 1.5
    return (11 + diff) / 8

# ==========================================
# 5. DYNAMIC ELO ENGINE (The Main Loop)
# Iterates through every match to update team strengths
# ==========================================
print("Running Elo Engine... This may take a moment.")
home_elo_pre, away_elo_pre = [], []

for idx, row in df_results.iterrows():
    h_team, a_team = row['home_team'], row['away_team']

    # Check if teams are new and fetch their starting points if necessary
    for team in [h_team, a_team]:
        if team not in elo_ratings:
            elo_ratings[team] = fetch_initial_points(team, row['date'])

    # Store ratings BEFORE the match (Essential for ML training)
    h_pre = elo_ratings[h_team]
    a_pre = elo_ratings[a_team]
    home_elo_pre.append(h_pre)
    away_elo_pre.append(a_pre)

    # Effective Home Rating: Add +100 for Home Advantage (if not neutral)
    h_elo_eff = h_pre + (100 if not row['neutral'] else 0)

    # Calculate Expected Result (E)
    exp_h = 1 / (1 + 10**((a_pre - h_elo_eff) / 400))

    # Actual Result (S): 1 for Win, 0.5 for Draw, 0 for Loss
    s_h = 1 if row['home_score'] > row['away_score'] else (0.5 if row['home_score'] == row['away_score'] else 0)

    # Calculate Elo Update
    k = get_k_factor(row['tournament'])
    g = get_goal_margin_factor(row['home_score'], row['away_score'])
    update = k * g * (s_h - exp_h)

    # Update Ratings in the dictionary
    elo_ratings[h_team] += update
    elo_ratings[a_team] -= update

# ==========================================
# 6. FEATURE ENGINEERING & CSV EXPORT
# ==========================================
df_results['home_elo'] = home_elo_pre
df_results['away_elo'] = away_elo_pre
df_results['elo_diff'] = df_results['home_elo'] - df_results['away_elo']

# Target Column: 0 = Home Win, 1 = Draw, 2 = Away Win
def get_target(h_s, a_s):
    if h_s > a_s: return 0
    if h_s == a_s: return 1
    return 2

df_results['target'] = df_results.apply(lambda x: get_target(x['home_score'], x['away_score']), axis=1)

# Export the final dataset for Machine Learning
output_file = 'final_training_data_2026.csv'
df_results.to_csv(output_file, index=False)

print("-" * 30)
print(f"Success! Final dataset ready: {output_file}")
print(f"Total unique teams processed: {len(elo_ratings)}")